# 02 · Postpartum Cardiometabolic Monitoring Prototype

**Edge Medicine Innovation Challenge · team prototype**

This notebook demonstrates the monitoring layer of the proposed platform in two parts.

| Part | Data | Status |
|---|---|---|
| A. Live platform check | JupyterHealth demo Exchange, study 30006 (synthetic Synthea CGM + Oura) | **Live** API data |
| B. Postpartum blood pressure monitoring | Simulated cohort of 400 patients with hypertensive disorders of pregnancy, calibrated to published programs | **Simulated** (the demo Exchange holds no blood pressure or pregnancy records) |

All simulated parameters are documented with sources in `pp_synth.py`. No real patient data is used anywhere in this notebook.

In [ ]:
import os
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
import ipywidgets as W
from IPython.display import HTML, Markdown, display

import pp_synth as ps

pd.options.mode.chained_assignment = None

try:
    from jupyterhealth_client import JupyterHealthClient, Code
    jh = JupyterHealthClient()
    LIVE = True
except Exception as e:  # running outside the Hub
    jh, LIVE = None, False
    print(f"Exchange not reachable ({type(e).__name__}); Part A will be skipped.")

STUDY_ID = 30006
PERSONA_ID = 40009   # Anika Patel (synthetic), b. 1989
print("Live Exchange connection:", LIVE)

## Part A · What the live demo data can support

Before building on study 30006, two checks establish what the data can and cannot be used for:

1. **Is each patient's CGM trace distinct?** Every patient shows exactly 1,345 glucose readings, which suggests a single trace replicated across the cohort. If so, the glucose data supports a single-patient display only, and any cross-patient glucose analysis would be invalid.
2. **What does one persona's monitoring picture look like?** Glucose metrics and nightly Oura signals for the persona, pulled through the same API a deployed system would use.

In [ ]:
if LIVE:
    sig = []
    for p in jh.list_patients(study_id=STUDY_ID):
        df = jh.list_observations_df(patient_id=p["id"], study_id=STUDY_ID,
                                     code=Code.BLOOD_GLUCOSE, limit=None)
        v = df.sort_values("effective_time_frame_date_time").blood_glucose_value.round(1)
        t = pd.to_datetime(df.effective_time_frame_date_time)
        sig.append({"patient_id": p["id"], "n": len(v),
                    "first": t.min(), "last": t.max(),
                    "mean": round(v.mean(), 1), "sd": round(v.std(), 1),
                    "trace_hash": hash(tuple(v))})
    cgm_check = pd.DataFrame(sig).set_index("patient_id")
    n_unique = cgm_check.trace_hash.nunique()
    display(Markdown(f"**Distinct CGM traces across {len(cgm_check)} patients: {n_unique}**"))
    if n_unique == 1:
        display(Markdown("All patients share one CGM trace. Use glucose for single-patient display only."))
    display(cgm_check.drop(columns="trace_hash"))

In [ ]:
if LIVE:
    g = jh.list_observations_df(patient_id=PERSONA_ID, study_id=STUDY_ID,
                                code=Code.BLOOD_GLUCOSE, limit=None)
    g = g.sort_values("effective_time_frame_date_time_local")
    v = g.blood_glucose_value
    metrics = {
        "Readings": f"{len(v):,}",
        "Span": f"{g.effective_time_frame_date_time_local.min():%b %d} to {g.effective_time_frame_date_time_local.max():%b %d}",
        "Mean glucose": f"{v.mean():.0f} mg/dL",
        "GMI": f"{3.31 + 0.02392 * v.mean():.2f}%",
        "CV": f"{v.std() / v.mean() * 100:.1f}%",
        "Time in range 70-140 (pregnancy/postpartum target)": f"{((v >= 70) & (v <= 140)).mean() * 100:.0f}%",
        "Time in range 70-180 (standard)": f"{((v >= 70) & (v <= 180)).mean() * 100:.0f}%",
    }
    display(Markdown("| Metric | Value |\n|---|---|\n" + "\n".join(f"| {k} | {m} |" for k, m in metrics.items())))
    px.line(g, x="effective_time_frame_date_time_local", y="blood_glucose_value",
            title=f"Persona {PERSONA_ID}: CGM trace (live Exchange data)",
            labels={"effective_time_frame_date_time_local": "", "blood_glucose_value": "mg/dL"}
            ).add_hrect(y0=70, y1=140, fillcolor="green", opacity=0.08, line_width=0).show()

In [ ]:
if LIVE:
    obs = jh.list_observations_df(patient_id=PERSONA_ID, study_id=STUDY_ID, limit=None)
    obs["night"] = pd.to_datetime(obs.get("effective_time_frame_time_interval_end_date_time")).dt.date
    wanted = {
        "omh:sleep-duration:2.0": ("sleep_duration_value", "Sleep (h)", 1 / 3600),
        "omh:heart-rate:2.0": ("heart_rate_value", "Sleeping HR (bpm)", 1),
        "omh:respiratory-rate:2.0": ("respiratory_rate_value", "Resp. rate (/min)", 1),
        "omh:step-count:3.0": ("step_count_value", "Steps", 1),
    }
    nightly = {}
    for c, (col, label, k) in wanted.items():
        sub = obs[obs.code == c]
        if col in sub:
            nightly[label] = sub.groupby("night")[col].mean() * k
    nightly = pd.DataFrame(nightly).sort_index()
    display(Markdown(f"**Persona {PERSONA_ID}: {len(nightly)} nights of Oura data, "
                     f"{nightly.index.min()} to {nightly.index.max()}**"))
    fig = px.line(nightly.reset_index().melt(id_vars="night"), x="night", y="value",
                  facet_row="variable", height=650, title="Nightly wearable signals (live Exchange data)")
    fig.update_yaxes(matches=None, title=None).for_each_annotation(lambda a: a.update(text=a.text.split("=")[-1]))
    fig.show()

**Reading Part A.** The live layer demonstrates the full data path the platform depends on: consented device data flowing through the Exchange's FHIR API into an analysis environment, in the same Open mHealth formats CareX, Dexcom, and Oura produce. In a postpartum deployment, overnight heart rate and sleep would act as context signals beside blood pressure; glucose would carry the gestational diabetes follow-up.

## Part B · Postpartum blood pressure monitoring (simulated cohort)

The simulated cohort enrolls every patient with a hypertensive disorder of pregnancy **at discharge by default**, the enrollment model the proposal argues for. Engagement then decays over time, at rates that differ by insurance and race in line with published attendance gaps.

In [ ]:
params = ps.CohortParams()
patients = ps.generate_patients(params)
readings = ps.generate_readings(patients, params)
readings["category"] = [ps.classify_bp_acog(s, d) for s, d in
                        zip(readings.systolic_blood_pressure_value, readings.diastolic_blood_pressure_value)]
summary = ps.engagement_summary(patients, readings)
engaged = summary[summary.n_readings > 0]

calib = pd.DataFrame([
    ("Hypertension after discharge (any reading >=140/90)", f"{engaged.any_hypertensive.mean():.0%}", "81.8%", "Hauspurg, JAMA Cardiol 2024"),
    ("Severe-range reading (>=160/110)", f"{engaged.any_severe.mean():.0%}", "~14%", "Hauspurg, JAMA Cardiol 2024"),
    ("Any reading within 10 days of discharge", f"{summary.ascertained_10d.mean():.0%}", "91.7% (RCT arm)", "Obstet Gynecol 2023;142:855"),
    ("Median days of use among users", f"{(engaged.last_day - engaged.first_day + 1).median():.0f}", "14", "Obstet Gynecol 2023;142:855"),
], columns=["Quantity", "Simulated", "Published", "Source"])
display(Markdown(f"**Cohort:** {len(patients)} patients, {len(readings):,} home readings"))
display(calib.style.hide(axis="index"))

### B1 · Who stays in the monitoring program?

The central equity question: even with universal enrollment, which patients are still being monitored when blood pressure peaks (days 3 to 6) and as the six-week window closes?

In [ ]:
def retention_curve(df, by):
    days = np.arange(0, ps.FOLLOW_UP_DAYS + 1)
    out = []
    for grp, sub in df.groupby(by):
        last = sub.last_day.fillna(-1)
        out += [{"day": d, by: grp, "still_monitored": (last >= d).mean()} for d in days]
    return pd.DataFrame(out)

tabs = W.Tab()
children = []
for by in ["insurance", "race_ethnicity", "language"]:
    out = W.Output()
    with out:
        fig = px.line(retention_curve(summary, by), x="day", y="still_monitored", color=by,
                      title=f"Share still sending readings, by {by.replace('_', ' ')}",
                      labels={"day": "Postpartum day", "still_monitored": ""})
        fig.update_yaxes(tickformat=".0%", range=[0, 1])
        fig.add_vrect(x0=3, x1=6, fillcolor="red", opacity=0.07, line_width=0,
                      annotation_text="BP peak", annotation_position="top left")
        fig.show()
        tbl = summary.groupby(by).agg(patients=("age", "size"),
                                      reading_within_10d=("ascertained_10d", "mean"),
                                      active_at_day_14=("still_active_day14", "mean"),
                                      severe_detected=("any_severe", "mean"))
        display(tbl.style.format({c: "{:.0%}" for c in tbl.columns if c != "patients"}))
    children.append(out)
tabs.children = children
for i, t in enumerate(["Insurance", "Race / ethnicity", "Language"]):
    tabs.set_title(i, t)
display(tabs)

### B2 · Patient view and primary care handoff

Select a patient to see her home readings against ACOG thresholds (140/90 hypertensive; 160/110 severe, requiring evaluation within the hour), her engagement record, and the automatically drafted handoff summary.

*Rule-based prototype logic for demonstration; it is not clinical decision support.*

In [ ]:
BAND = {"controlled": "#4a74b4", "hypertensive": "#e6a700", "severe": "#d4322c"}

def handoff_card(pid):
    s = summary.loc[pid]
    r = readings[readings.patient_id == pid]
    n_sev = int((r.category == "severe").sum())
    if s.n_readings == 0:
        action = "No readings received. Outreach needed: confirm cuff delivery and phone access."
    elif n_sev:
        action = f"{n_sev} severe-range reading(s). Same-day clinical review."
    elif s.last_day < 10:
        action = f"Stopped sending readings on day {int(s.last_day)}. Outreach before the 7 to 10 day BP check window closes."
    elif s.any_hypertensive:
        action = "Persistent hypertension. Primary care follow-up for antihypertensive management."
    else:
        action = "Readings controlled. Routine follow-up; annual cardiometabolic screening."
    rows = [
        ("Diagnosis", s.hdp_subtype.replace("_", " ")),
        ("Discharged", f"{s.discharge_date:%b %d, %Y}"),
        ("Readings received", f"{s.n_readings} on {s.days_with_reading} days"),
        ("Monitoring window", "none" if s.n_readings == 0 else f"day {int(s.first_day)} to day {int(s.last_day)}"),
        ("Reading within 10 days", "yes" if s.ascertained_10d else "no"),
        ("Highest systolic", "n/a" if s.n_readings == 0 else f"{int(s.max_sbp)} mmHg"),
        ("Insurance / language", f"{s.insurance} / {s.language}"),
    ]
    body = "".join(f"<tr><td style='padding:3px 12px 3px 0;color:#666'>{k}</td><td><b>{v}</b></td></tr>" for k, v in rows)
    return HTML(f"""<div style='border:1px solid #ccc;border-radius:8px;padding:14px 18px;max-width:560px;font-family:sans-serif'>
      <div style='font-size:15px;font-weight:600;margin-bottom:6px'>Postpartum handoff · {pid}</div>
      <table style='font-size:13px'>{body}</table>
      <div style='margin-top:10px;padding:8px 10px;background:#f6f6f6;border-left:4px solid #d4322c;font-size:13px'>
      <b>Suggested next step:</b> {action}</div></div>""")

def plot_patient(pid):
    r = readings[readings.patient_id == pid]
    fig = go.Figure()
    if len(r):
        fig.add_trace(go.Scatter(x=r.postpartum_day, y=r.systolic_blood_pressure_value, mode="lines+markers",
                                 name="Systolic", line=dict(color="#333"),
                                 marker=dict(color=[BAND[c] for c in r.category], size=9)))
        fig.add_trace(go.Scatter(x=r.postpartum_day, y=r.diastolic_blood_pressure_value, mode="lines+markers",
                                 name="Diastolic", line=dict(color="#999", dash="dot"),
                                 marker=dict(color=[BAND[c] for c in r.category], size=7)))
    for y, txt, col in [(140, "140 hypertensive", "#e6a700"), (160, "160 severe", "#d4322c"),
                        (90, "90", "#e6a700"), (110, "110 severe", "#d4322c")]:
        fig.add_hline(y=y, line_dash="dash", line_color=col, opacity=0.6, annotation_text=txt)
    fig.add_vrect(x0=7, x1=10, fillcolor="green", opacity=0.07, line_width=0,
                  annotation_text="ACOG 7-10 day check", annotation_position="bottom left")
    fig.update_layout(title=f"{pid}: home blood pressure after discharge", height=420,
                      xaxis=dict(title="Postpartum day", range=[0, ps.FOLLOW_UP_DAYS + 1]),
                      yaxis=dict(title="mmHg", range=[50, 190]))
    fig.show()
    display(handoff_card(pid))

# Default to an informative case: an early dropout who had a hypertensive reading.
default = summary[(summary.n_readings > 0) & (summary.last_day < 10) & summary.any_hypertensive].index
options = list(summary.sort_values(["any_severe", "n_readings"], ascending=False).index)
W.interact(plot_patient, pid=W.Dropdown(options=options, value=default[0] if len(default) else options[0],
                                        description="Patient"));

### B3 · Export (optional)

Writes the simulated readings as Open mHealth `blood-pressure:4.0` records, the format the Exchange's CSV Upload source and CareX produce. Output goes to `data/`, which git ignores.

In [ ]:
os.makedirs("data", exist_ok=True)
import json
with open("data/synthetic_postpartum_bp_omh.jsonl", "w") as f:
    for row in readings.itertuples():
        f.write(json.dumps(ps.to_omh_record(row)) + "\n")
patients.drop(columns=[c for c in patients if c.endswith("_true") or c == "sbp_patient_level"]
              ).to_csv("data/synthetic_postpartum_patients.csv", index=False)
print("Wrote", len(readings), "OMH records and", len(patients), "patients to data/")